In [2]:
import pandas as pd

disease_files = {
    "cardiovascular": "cardiovascular_full_dataset_for_tableau.csv",
    "kidney": "kidney_full_dataset_for_tableau.csv",
    "diabetes": "diabetes_full_dataset_for_tableau.csv",
    "liver": "liver_full_dataset_for_tableau.csv",
    "obesity": "obesity_full_dataset_for_tableau.csv",
    "autoimmune": "autoimmune_full_dataset_for_tableau.csv",
    "rare_disease": "rare_disease_full_dataset_for_tableau.csv",
}

dfs = []
for area, filename in disease_files.items():
    df = pd.read_csv(filename, keep_default_na=False)
    df["disease_area"] = area
    dfs.append(df)

df_combined = pd.concat(dfs, ignore_index=True)
print(df_combined.shape)
print(df_combined["disease_area"].value_counts())

(9408, 22)
disease_area
cardiovascular    3023
autoimmune        1559
diabetes          1426
kidney            1309
liver             1168
obesity            840
rare_disease        83
Name: count, dtype: int64


In [3]:
true_labels = ["true_cv", "true_kidney", "true_diabetes", "true_liver",
               "true_obesity", "true_autoimmune", "true_rare_disease"]

df_true_combined = df_combined[df_combined["indication_classification"].isin(true_labels)]
print(df_true_combined.shape)
print(df_true_combined["disease_area"].value_counts())

(6124, 22)
disease_area
cardiovascular    1742
autoimmune        1238
liver              867
diabetes           846
kidney             791
obesity            586
rare_disease        54
Name: count, dtype: int64


In [4]:
df_combined.to_csv("all_disease_areas_combined_for_tableau.csv", index=False)

## Sponsor Footprint

In [5]:
sponsor_footprint = (
    df_true_combined.groupby("dashboard_sponsor_name")["disease_area"]
    .nunique()
    .sort_values(ascending=False)
)
sponsor_footprint.to_csv("sponsor_disease_area_footprint.csv")
print(sponsor_footprint.head(20))

dashboard_sponsor_name
Roche                                               7
Novo Nordisk                                        7
AstraZeneca                                         7
Novartis                                            7
Innovent Biologics (Suzhou) Co. Ltd.                6
Amgen                                               6
Eli Lilly                                           6
Regeneron                                           6
Pfizer                                              6
Boehringer Ingelheim                                6
AbbVie                                              5
Hanmi Pharmaceutical Company Limited                5
Jiangsu Hansoh Pharmaceutical Co., Ltd.             5
Chia Tai Tianqing Pharmaceutical Group Co., Ltd.    5
Jiangsu HengRui Medicine Co., Ltd.                  5
Qilu Pharmaceutical Co., Ltd.                       5
Sanofi                                              5
Takeda                                              5
Merck

## Trial Volume Per Sponsor Per Disease Area

In [6]:
sponsor_by_disease = (
    df_true_combined.groupby(["dashboard_sponsor_name", "disease_area"])["nct_id"]
    .nunique()
    .reset_index()
    .rename(columns={"nct_id": "trial_count"})
)
sponsor_by_disease.to_csv("sponsor_trial_count_by_disease_area.csv", index=False)

In [8]:
df_indication_combined = df_true_combined.assign(
    conditions=df_true_combined["conditions"].str.split("|")
).explode("conditions")

indication_windows = (
    df_indication_combined.groupby(["conditions", "date_window"])["nct_id"]
    .nunique()
    .unstack(fill_value=0)
)
indication_windows.columns = ["window1", "window2"]

indication_windows["total_volume"] = indication_windows["window1"] + indication_windows["window2"]
indication_windows["pct_change"] = (
    (indication_windows["window2"] - indication_windows["window1"]) / indication_windows["window1"]
).where(indication_windows["window1"] > 0)

# Filter out noise the same way you did per-disease: require meaningful volume
indication_windows_filtered = indication_windows[indication_windows["total_volume"] >= 5]

best = indication_windows_filtered.sort_values("pct_change", ascending=False).head(5)
worst = indication_windows_filtered.sort_values("pct_change", ascending=True).head(5)

print("TOP 5 GROWING:")
print(best)
print("\nTOP 5 DECLINING:")
print(worst)

TOP 5 GROWING:
                                                window1  window2  \
conditions                                                         
Obesity & Overweight                                  1       16   
SLE                                                   1       10   
Weight Management                                     1       10   
Atherosclerotic Cardiovascular Disease (ASCVD)        1        9   
Lupus Nephritis (LN)                                  1        9   

                                                total_volume  pct_change  
conditions                                                                
Obesity & Overweight                                      17        15.0  
SLE                                                       11         9.0  
Weight Management                                         11         9.0  
Atherosclerotic Cardiovascular Disease (ASCVD)            10         8.0  
Lupus Nephritis (LN)                                      

## Sanity Check

In [10]:
disease_windows = (
    df_true_combined.groupby(["disease_area", "date_window"])["nct_id"]
    .nunique()
    .unstack(fill_value=0)
)
disease_windows.columns = ["window1", "window2"]
disease_windows["pct_change"] = (
    (disease_windows["window2"] - disease_windows["window1"]) / disease_windows["window1"]
)
print(disease_windows.sort_values("pct_change"))

                window1  window2  pct_change
disease_area                                
liver               509      358   -0.296660
cardiovascular      956      786   -0.177824
diabetes            460      386   -0.160870
kidney              430      361   -0.160465
rare_disease         27       27    0.000000
autoimmune          593      645    0.087690
obesity             172      414    1.406977


In [11]:
disease_windows.to_csv("disease_area_trial_volume_pct_change.csv")

In [12]:
obesity_sponsors = (
    df_true_combined[df_true_combined["disease_area"] == "obesity"]
    .groupby(["dashboard_sponsor_name", "date_window"])["nct_id"]
    .nunique()
    .unstack(fill_value=0)
)
obesity_sponsors.columns = ["window1", "window2"]
obesity_sponsors = obesity_sponsors.sort_values("window2", ascending=False)

total_window2 = obesity_sponsors["window2"].sum()
top3_window2 = obesity_sponsors["window2"].head(3).sum()
top3_share = top3_window2 / total_window2

print(obesity_sponsors.head(10))
print(f"\nTotal obesity trials in Window 2: {total_window2}")
print(f"Top 3 sponsors' share of Window 2: {top3_share:.1%}")

# How many distinct sponsors have ANY obesity activity in window 2?
active_sponsors_w2 = (obesity_sponsors["window2"] > 0).sum()
print(f"Total distinct sponsors active in Window 2: {active_sponsors_w2}")

                                                    window1  window2
dashboard_sponsor_name                                              
Novo Nordisk                                             39       66
Eli Lilly                                                27       51
Amgen                                                     2       14
Fujian Shengdi Pharmaceutical Co., Ltd.                   2       14
Pfizer                                                    7       12
Boehringer Ingelheim                                      7       12
Hangzhou Zhongmei Huadong Pharmaceutical Co., Ltd.        0       10
BrightGene Bio-Medical Technology Co., Ltd.               0        9
Gan & Lee Pharmaceuticals.                                0        8
Beijing QL Biopharmaceutical Co.,Ltd                      0        7

Total obesity trials in Window 2: 414
Top 3 sponsors' share of Window 2: 31.6%
Total distinct sponsors active in Window 2: 126


In [13]:
obesity_sponsors.to_csv("obesity_sponsor_concentration.csv")